⏱️ **Time required:** ~10 minutes | **Type:** Interactive tutorial

# Engine Portability & Scale

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LakeLogic/LakeLogic/blob/main/examples/colab/03_engine_scale.ipynb) [![View on GitHub](https://img.shields.io/badge/github-view_source-black?logo=github)](https://github.com/lakelogic/LakeLogic/blob/main/examples/colab/03_engine_scale.ipynb)

Write once, run anywhere. Same contract, different engines, identical results — plus dimensional modeling, incremental processing, parallel execution, backfill, and external logic hooks.

In [16]:
# Install lakelogic
!pip install -q lakelogic[polars,duckdb]

import urllib.request
import os

if not os.path.exists("_setup.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/LakeLogic/LakeLogic/main/examples/colab/_setup.py", "_setup.py"
    )
import _setup as s
import lakelogic as ll

### ⚙️ Execution Engine

In [ ]:
# Select execution engine (Colab comes with PySpark pre-installed)
ENGINE = "duckdb" # 'polars' , 'spark'

---
## 1. Engine-Agnostic Proof — Zero Code Changes

**The Problem:** You built your pipeline on Polars/DuckDB. Now it needs to run on Spark in production. Rewriting 2,000 lines of DataFrame logic isn't a weekend project.

**The Solution:** LakeLogic compiles SQL-first rules to each engine's dialect. Same contract, same results.

In [17]:
import os
import shutil

# ── Clean slate: remove stale files from previous runs ────────────
for _f in ["03_engine_scale_demo/engine_test.yaml", "engine_test_source.parquet"]:
    if os.path.exists(_f):
        os.remove(_f)

contract = s.write_contract(
    """
version: 1.0.0
dataset: engine_test

model:
  fields:
    - name: id
      type: integer
      required: true
    - name: email
      type: string
      required: true
    - name: score
      type: integer

quality:
  row_rules:
    - name: valid_email
      sql: "email LIKE '%@%.%'"
    - name: score_range
      sql: "score BETWEEN 0 AND 100"
""",
    "03_engine_scale_demo/engine_test.yaml",
)

source_df = ll.DataGenerator(contract).generate(rows=500, invalid_ratio=0.08, output_format=ENGINE)
source_file = "engine_test_source.parquet"
source_df.write_parquet(source_file)

# Run on Polars
p1 = ll.DataProcessor(contract, engine=ENGINE)
r1 = p1.run(source_df)

# Run on DuckDB — same contract, zero changes
p2 = ll.DataProcessor(contract, engine=ENGINE)
r2 = p2.run(source_df)

# Run on Spark — same contract, zero changes -
run_spark = False
if run_spark:
    p3 = ll.DataProcessor(contract, engine=ENGINE)
    from pyspark.sql import SparkSession

    spark = SparkSession.builder.appName("LakeLogic").getOrCreate()
    spark_df = spark.read.parquet(source_file)
    r3 = p3.run(spark_df)

2026-04-28 06:49:07.752 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: engine_test
2026-04-28 06:49:07.753 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 460 valid + 40 invalid = 500 total
2026-04-28 06:49:07.753 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:49:07.754 | INFO     | lakelogic.core.generator:generate:3448 -    Edge cases : Heuristic-only (no AI edge cases available)
2026-04-28 06:49:07.821 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 500 records built
2026-04-28 06:49:07.821 | INFO     | lakelogic.core.generator:generate:3504 -    Test cases : 53 across 5 categories
2026-04-28 06:49:07.821 | INFO     | lakelogic.core.generator:generate:3506 -      NOT_NULL_VIOLATION               19 injections
2026-04-28 06:49:07.822 | INFO     | lakelogic.core.generator:generate:3506 -      RANGE_VIOLATION      

In [18]:
# The Proof
print("Engine Comparison")
print("=" * 40)
print(f"  Polars : good={r1.good_count}, bad={r1.bad_count}")
print(f"  DuckDB : good={r2.good_count}, bad={r2.bad_count}")

if run_spark:
    print(f"  Spark : good={r3.good_count}, bad={r3.bad_count}")
    print(
        f"  Match  : {r1.good_count == r2.good_count and r1.bad_count == r2.bad_count and r2.good_count == r3.good_count and r2.bad_count == r3.bad_count}"
    )

else:
    print(f"  Match  : {r1.good_count == r2.good_count and r1.bad_count == r2.bad_count}")

print("\nSame contract. Same data. Same results. Zero code changes.")

Engine Comparison
  Polars : good=7, bad=493
  DuckDB : good=7, bad=493
  Match  : True

Same contract. Same data. Same results. Zero code changes.


---
## 2. Dimensional Modeling — SCD2, Merge, Overwrite

**The Problem:** Your dimension table needs history tracking. You manually build `MERGE INTO` SQL, manage `effective_from`/`effective_to` dates, and debug `is_current` flags by hand.

**The Solution:** Declare `materialization.strategy: scd2` in the contract — LakeLogic generates all SCD2 columns, applies merge logic, and manages version tracking. Zero SQL required.

In [19]:
import yaml

# ── Show what a fully-declared dimensional contract looks like ────────
scd2_yaml = """
version: 1.0.0
dataset: dim_customers
info:
  title: gold_dim_customers
  target_layer: gold

primary_key: [customer_id]

model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email
      type: string
    - name: tier
      type: string

materialization:
  strategy: scd2
  scd2:
    track_columns: [name, email, tier]
    timestamp_field: updated_at
    surrogate_key: _sk
    effective_from_field: effective_from
    effective_to_field: effective_to
    current_flag_field: is_current
    end_date_default: "9999-12-31"
    version_column: "_version"             # ROW_NUMBER per business key
    change_reason_column: "_change_reason" # "initial_load", "email,tier changes", etc.

    # Unknown member (late-arriving fact fallback)
    unknown_member:
      enabled: true
      surrogate_key_value: "-1"
"""

# Parse & validate the contract
from lakelogic.core.models import DataContract

scd2_contract = ll.DataContract(**yaml.safe_load(scd2_yaml))

print("\u2500" * 60)
print("DIMENSIONAL MODELING: SCD2 Contract")
print("\u2500" * 60)
print(f"  Strategy       : {scd2_contract.materialization.strategy}")
print(f"  Primary key    : {scd2_contract.primary_key}")
print(f"  Track columns  : {scd2_contract.materialization.scd2['track_columns']}")
print(f"  Surrogate key  : {scd2_contract.materialization.scd2['surrogate_key']}")
print(f"  Effective from : {scd2_contract.materialization.scd2['effective_from_field']}")
print(f"  Effective to   : {scd2_contract.materialization.scd2['effective_to_field']}")
print(f"  Current flag   : {scd2_contract.materialization.scd2['current_flag_field']}")
print(f"  version        : {scd2_contract.materialization.scd2['version_column']}")
print(f"  change reason  : {scd2_contract.materialization.scd2['change_reason_column']}")
print(f"  unknown_member : {scd2_contract.materialization.scd2['unknown_member']}")


# ── Show all supported strategies ─────────────────────────────────────
strategies = {
    "append": "Fact tables — new rows added, never updated",
    "merge": "SCD Type 1 — upsert by natural key, latest value wins",
    "scd2": "SCD Type 2 — full history with effective dates",
    "overwrite": "Periodic snapshot — drop & replace on each run",
}

print("\n" + "\u2500" * 60)
print("ALL MATERIALIZATION STRATEGIES")
print("\u2500" * 60)
for strat, desc in strategies.items():
    marker = "\u2716" if strat == scd2_contract.materialization.strategy else " "
    print(f"  [{marker}] {strat:10s} — {desc}")
print("\n\u2705 All declared in YAML. No manual MERGE INTO SQL required.")

────────────────────────────────────────────────────────────
DIMENSIONAL MODELING: SCD2 Contract
────────────────────────────────────────────────────────────
  Strategy       : scd2
  Primary key    : ['customer_id']
  Track columns  : ['name', 'email', 'tier']
  Surrogate key  : _sk
  Effective from : effective_from
  Effective to   : effective_to
  Current flag   : is_current
  version        : _version
  change reason  : _change_reason
  unknown_member : {'enabled': 'true', 'surrogate_key_value': '-1'}

────────────────────────────────────────────────────────────
ALL MATERIALIZATION STRATEGIES
────────────────────────────────────────────────────────────
  [ ] append     — Fact tables — new rows added, never updated
  [ ] merge      — SCD Type 1 — upsert by natural key, latest value wins
  [✖] scd2       — SCD Type 2 — full history with effective dates
  [ ] overwrite  — Periodic snapshot — drop & replace on each run

✅ All declared in YAML. No manual MERGE INTO SQL required.


### The Proof — Let's see SCD2 in action
We'll generate 3 rows, run them through the processor, and see how LakeLogic automatically injects and populates the tracking columns without any SQL.

In [20]:
import os
import glob
import polars as pl

# ── Clean slate: remove stale files from previous runs ────────────
for _target in ["03_engine_scale_demo/parallel_demo/gold_dim_customers", "03_engine_scale_demo/dim_customers.yaml"]:
    if os.path.isdir(_target):
        shutil.rmtree(_target)
    elif os.path.isfile(_target):
        os.remove(_target)

# Generate some initial data
scd2_path = s.write_contract(scd2_yaml, "03_engine_scale_demo/dim_customers.yaml")
scd2_source = ll.DataGenerator(scd2_path).generate(rows=3, output_format=ENGINE)
print("1. INCOMING SOURCE DATA:\n")
display(scd2_source)

# Run the pipeline (LakeLogic automatically handles the SCD2 merge logic)
p_scd2 = ll.DataProcessor(scd2_path, engine=ENGINE)
os.makedirs("03_engine_scale_demo/parallel_demo/gold_dim_customers", exist_ok=True)
good_scd2, bad_scd2 = p_scd2.run(scd2_source, materialize=True, materialize_target="parallel_demo/gold_dim_customers")

print("\n2. AFTER LAKELOGIC SCD2 PROCESSING (Notice the injected tracking columns):\n")
try:
    materialized_df = pl.read_delta("03_engine_scale_demo/parallel_demo/gold_dim_customers")
except Exception:
    from pathlib import Path

    search_path = Path("03_engine_scale_demo/parallel_demo/gold_dim_customers")
    parquet_files = [str(p) for p in search_path.rglob("*.parquet")]
    if parquet_files:
        materialized_df = pl.read_parquet(parquet_files)
    else:
        materialized_df = good_scd2
display(materialized_df.select([c for c in materialized_df.columns if not c.startswith("_lakelogic_")]))

2026-04-28 06:49:07.953 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: gold_dim_customers
2026-04-28 06:49:07.955 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 3 valid + 0 invalid = 3 total
2026-04-28 06:49:07.955 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:49:07.958 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 3 records built


1. INCOMING SOURCE DATA:



customer_id,name,email,tier,_is_invalid
i64,str,str,str,bool
6964,"""Colleen Ward""","""sandersmelanie@example.net""","""enterprise""",false
3135,"""Nathan Pruitt""","""valenciatara@example.org""","""free""",false
1346,"""Cynthia Petersen""","""perezlisa@example.com""","""free""",false


2026-04-28 06:49:07.973 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=gold] | Source: 3 | Total: 3 | Good: 3 | Quarantine: 0 | Ratio: 0.00%
2026-04-28 06:49:07.974 | WARNING  | lakelogic.core.processor:run:1026 - Schema drift detected for 'gold_dim_customers': missing=[], unknown=['_is_invalid']
2026-04-28 06:49:07.989 | INFO     | lakelogic.core.materialization:_inject_unknown_member_pandas:1292 - Injected unknown member row (SK=-1) into dimension
2026-04-28 06:49:07.996 | INFO     | lakelogic.core.materialization:materialize_dataframe:3708 - Materialized 4 rows to 03_engine_scale_demo\parallel_demo\gold_dim_customers\data.parquet



2. AFTER LAKELOGIC SCD2 PROCESSING (Notice the injected tracking columns):



customer_id,name,email,tier,_is_invalid,effective_from,effective_to,is_current,_change_reason,_sk,_version
i64,str,str,str,i64,str,str,bool,str,str,i64
1346,"""Cynthia Petersen""","""perezlisa@example.com""","""free""",0,"""1900-01-01""","""9999-12-31""",true,"""initial_load""","""7fb8a3d0ffbe0000""",1
3135,"""Nathan Pruitt""","""valenciatara@example.org""","""free""",0,"""1900-01-01""","""9999-12-31""",true,"""initial_load""","""8af047ce89bcf2ff""",1
6964,"""Colleen Ward""","""sandersmelanie@example.net""","""enterprise""",0,"""1900-01-01""","""9999-12-31""",true,"""initial_load""","""b548a1567c31c40d""",1
-1,"""Unknown""","""Unknown""","""Unknown""",-1,"""1900-01-01""","""9999-12-31""",true,"""unknown_member""","""-1""",0


---
## 3. Incremental Processing — `pipeline_log` Watermark

**The Problem:** Your nightly job reprocesses 10 million rows even though only 500 changed. Compute costs scale with total volume instead of change volume.

**The Solution:** LakeLogic's `pipeline_log` watermark strategy tracks which files have been processed by their modification time. On the next run, only **new files** are loaded.

In [21]:
import os
import shutil
import polars as pl

# ── Clean slate for demo ─────────────────────────────────────────────
DEMO_DIR = "./incremental_demo"
LANDING = f"{DEMO_DIR}/landing"

if os.path.exists(DEMO_DIR):
    shutil.rmtree(DEMO_DIR)
os.makedirs(LANDING, exist_ok=True)

# ── Contract with source.type = landing, load_mode = incremental ────
inc_contract = s.write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_orders
  target_layer: bronze

source:
  type: landing
  path: "./incremental_demo/landing"
  format: ndjson
  load_mode: incremental
  watermark_strategy: pipeline_log

metadata:
  run_log_dir: "./incremental_demo/logs"

model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
    - name: status
      type: string

quality:
  row_rules:
    - name: positive_amount
      sql: "amount > 0"
""",
    "03_engine_scale_demo/orders_inc.yaml",
)

# ── FILE 1: 50 orders land in the landing zone ───────────────────────
batch1 = ll.DataGenerator(inc_contract).generate(rows=50, output_format=ENGINE)
batch1.write_ndjson(f"{LANDING}/orders_batch_1.json")
print(f"\u2705 File 1: wrote {len(batch1)} rows to orders_batch_1.json")

# ── RUN 1: Initial load (no prior watermark) ────────────────────────
proc = ll.DataProcessor(inc_contract, engine=ENGINE)
g1, b1 = proc.run_source()
g1, b1 = s.to_polars(g1), s.to_polars(b1)
r1 = proc.last_report

print(f"\n{'=' * 50}")
print("RUN 1 (initial load)")
print(f"{'=' * 50}")
print(f"  Files in landing : {len(os.listdir(LANDING))}")
print(f"  Rows loaded      : {r1.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r1.get('counts', {}).get('good', '?')} / {r1.get('counts', {}).get('quarantined', '?')}")

2026-04-28 06:49:08.021 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: bronze_orders
2026-04-28 06:49:08.022 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 50 valid + 0 invalid = 50 total
2026-04-28 06:49:08.022 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:49:08.023 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 50 records built
2026-04-28 06:49:08.031 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\incremental_demo\landing via polars


2026-04-28 06:49:08.037 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 50 | Total: 50 | Good: 48 | Quarantine: 2 | Ratio: 4.00%
2026-04-28 06:49:08.038 | WARNING  | lakelogic.core.processor:run:1026 - Schema drift detected for 'bronze_orders': missing=[], unknown=['_is_invalid']


✅ File 1: wrote 50 rows to orders_batch_1.json

RUN 1 (initial load)
  Files in landing : 1
  Rows loaded      : 50
  Good / Bad       : 48 / 2


In [22]:
import time

time.sleep(1)  # Ensure mtime of File 2 is strictly after Run 1's watermark

# ── FILE 2: 20 new orders arrive ────────────────────────────────────
batch2 = ll.DataGenerator(inc_contract).generate(rows=20, output_format=ENGINE)
batch2.write_ndjson(f"{LANDING}/orders_batch_2.json")
print(f"\u2705 File 2: wrote {len(batch2)} rows to orders_batch_2.json")
print(f"  Landing zone now has: {os.listdir(LANDING)}")

# ── RUN 2: Only new files processed ─────────────────────────────────
g2, b2 = proc.run_source()
g2, b2 = s.to_polars(g2), s.to_polars(b2)
r2 = proc.last_report

print(f"\n{'=' * 50}")
print("RUN 2 (incremental)")
print(f"{'=' * 50}")
print(f"  Files in landing : {len(os.listdir(LANDING))} (70 total rows across 2 files)")
print("  Files processed  : 1 (only orders_batch_2.json \u2014 batch_1 already processed)")
print(f"  Rows loaded      : {r2.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r2.get('counts', {}).get('good', '?')} / {r2.get('counts', {}).get('quarantined', '?')}")
print("\n\u2705 pipeline_log watermark: only new files are processed. No reprocessing.")

2026-04-28 06:49:09.054 | INFO     | lakelogic.core.generator:generate:3416 - 📋 Generating data for: bronze_orders
2026-04-28 06:49:09.055 | INFO     | lakelogic.core.generator:generate:3417 -    Records    : 20 valid + 0 invalid = 20 total
2026-04-28 06:49:09.055 | INFO     | lakelogic.core.generator:generate:3433 -    Source     : Faker + heuristic generation (no AI or file seeds)
2026-04-28 06:49:09.056 | INFO     | lakelogic.core.generator:generate:3482 -    Row generation complete: 20 records built
2026-04-28 06:49:09.059 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: C:\_Personal\_SaaS\lakelogic\examples\colab\incremental_demo\landing via polars
2026-04-28 06:49:09.068 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 70 | Total: 70 | Good: 68 | Quarantine: 2 | Ratio: 2.86%
2026-04-28 06:49:09.069 | WARNING  | lakelogic.core.processor:run:1026 - Schema drift detected for 'bronze_orders': missing=[], unknown=['_is_invali

✅ File 2: wrote 20 rows to orders_batch_2.json
  Landing zone now has: ['orders_batch_1.json', 'orders_batch_2.json']

RUN 2 (incremental)
  Files in landing : 2 (70 total rows across 2 files)
  Files processed  : 1 (only orders_batch_2.json — batch_1 already processed)
  Rows loaded      : 70
  Good / Bad       : 68 / 2

✅ pipeline_log watermark: only new files are processed. No reprocessing.


---
## 4. Parallel Processing — Concurrent Multi-Contract Execution

**The Problem:** You have 8 Bronze contracts with no dependencies between them. Running them sequentially takes 40 minutes.

**The Solution:** `pipeline.run(parallel=True)` groups contracts into dependency **waves** using topological sort. Contracts within the same wave execute concurrently via threads — layer ordering is preserved automatically.

In [23]:
import os
import shutil
import yaml
from lakelogic.core.registry import DomainRegistry
from lakelogic.pipeline import LakehousePipeline
from IPython.display import HTML, display

# ── Create inline contracts ──────────────────────────────────────────
DAG_DIR = "./parallel_demo"

# ── Clean slate: remove stale files from previous runs ────────────
if os.path.exists(DAG_DIR):
    shutil.rmtree(DAG_DIR)

os.makedirs(f"{DAG_DIR}/contracts/bronze", exist_ok=True)
os.makedirs(f"{DAG_DIR}/contracts/silver", exist_ok=True)

# Bronze: orders (independent)
s.write_contract(
    """
version: 1.0.0
dataset: orders
info:
  title: bronze_orders
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/orders"
  format: ndjson
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
""",
    f"{DAG_DIR}/contracts/bronze/orders.yaml",
)

# Bronze: customers (independent — runs in parallel with orders)
s.write_contract(
    """
version: 1.0.0
dataset: customers
info:
  title: bronze_customers
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/customers"
  format: ndjson
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email_address
      type: string
      pii: true
""",
    f"{DAG_DIR}/contracts/bronze/customers.yaml",
)

# Bronze: products (independent — runs in parallel with orders & customers)
s.write_contract(
    """
version: 1.0.0
dataset: products
info:
  title: bronze_products
  target_layer: bronze
source:
  type: landing
  path: "./parallel_demo/landing/products"
  format: ndjson
model:
  fields:
    - name: product_id
      type: integer
      required: true
    - name: name
      type: string
""",
    f"{DAG_DIR}/contracts/bronze/products.yaml",
)

# Silver: customers_enriched
s.write_contract(
    """
version: 1.0.0
dataset: customers_enriched
info:
  title: silver_customers_enriched
  target_layer: silver
source:
  type: table
  path: "./parallel_demo/lakehouse/bronze/bronze_customers"
model:
  fields:
    - name: customer_id
      type: integer
      required: true
    - name: name
      type: string
    - name: email_address
      type: string
      pii: true

""",
    f"{DAG_DIR}/contracts/silver/customers_enriched.yaml",
)

# Silver: orders_enriched (depends on orders + customers → runs AFTER them)
s.write_contract(
    """
version: 1.0.0
dataset: orders_enriched
info:
  title: silver_orders_enriched
  target_layer: silver
source:
  type: table
  path: "./parallel_demo/lakehouse/bronze/bronze_orders"
model:
  fields:
    - name: order_id
      type: integer
      required: true
    - name: amount
      type: float
downstream:
  - type: dashboard
    name: "Weekly Sales Performance"
    platform: power_bi
    url: "https://app.powerbi.com/..."
    owner: "marketing-analytics"

  - type: api
    name: "Order Tracking Service"
    platform: internal
    owner: "backend-team"

""",
    f"{DAG_DIR}/contracts/silver/orders_enriched.yaml",
)

# ── Create _system.yaml with dependency declarations ────────────────
system_yaml = {
    "domain": "demo",
    "system": "ecommerce",
    # ── External sources (for lineage visualization) ──────────────────────────
    "external_sources": [
        {
            "name": "Shopify API",
            "source_domain": "CRM Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["orders", "customers"],
        },
        {
            "name": "Products System Database",
            "source_domain": "Products Vendor",
            "catalog_path": "external_storage_path_or_api",
            "consumed_by": ["products"],
        },
    ],
    "contracts": [
        {"layer": "bronze", "entity": "orders", "path": "contracts/bronze/orders.yaml", "enabled": True},
        {"layer": "bronze", "entity": "customers", "path": "contracts/bronze/customers.yaml", "enabled": True},
        {"layer": "bronze", "entity": "products", "path": "contracts/bronze/products.yaml", "enabled": True},
        {
            "layer": "silver",
            "entity": "customers_enriched",
            "path": "contracts/silver/customers_enriched.yaml",
            "enabled": True,
        },
        {
            "layer": "silver",
            "entity": "orders_enriched",
            "path": "contracts/silver/orders_enriched.yaml",
            "depends_on": ["customers_enriched"],
            "enabled": True,
        },
    ],
    "environments": {
        "local": {
            "catalog": "local",
            "storage_root": "./parallel_demo/lakehouse",
            "data_root": "./parallel_demo/lakehouse",
            "quarantine_root": "./parallel_demo/lakehouse/_quarantine",
        }
    },
    "storage": {"external_location_root": "./parallel_demo/lakehouse"},
}

sys_path = f"{DAG_DIR}/_system.yaml"
with open(sys_path, "w") as f:
    yaml.dump(system_yaml, f, default_flow_style=False)

# ── Build pipeline ──────────────────────────────────────────────────
registry = DomainRegistry.from_yaml(sys_path, environment="local", storage_mode="direct")
pipeline = LakehousePipeline(registry, engine=ENGINE)

# ── Visualise the DAG — shows parallel waves ────────────────────────
display(HTML(pipeline.visualize_dag()))

# ── Show wave grouping ──────────────────────────────────────────────
from lakelogic.pipeline.runner import LakehousePipeline as _LP

bronze_contracts = [c for c in registry.contracts if c.layer == "bronze"]
waves = _LP._group_by_dependency_level(bronze_contracts)

print("\n" + "\u2500" * 60)
print("PARALLEL EXECUTION PLAN")
print("\u2500" * 60)
print(f"  Bronze layer: {len(bronze_contracts)} contracts")
for i, wave in enumerate(waves):
    entities = [c.entity for c in wave]
    print(f"  Wave {i}: [{', '.join(entities)}] \u2190 {'parallel' if len(entities) > 1 else 'sequential'}")

print("\n  Silver layer: orders_enriched")
print("  \u2514\u2500 depends_on: [orders, customers] \u2192 waits for Bronze to complete")

print("\n\u2705 pipeline.run(parallel=True) executes Wave 0 contracts concurrently.")
print("   Layer ordering (bronze \u2192 silver \u2192 gold) is always preserved.")


────────────────────────────────────────────────────────────
PARALLEL EXECUTION PLAN
────────────────────────────────────────────────────────────
  Bronze layer: 3 contracts
  Wave 0: [orders, customers, products] ← parallel

  Silver layer: orders_enriched
  └─ depends_on: [orders, customers] → waits for Bronze to complete

✅ pipeline.run(parallel=True) executes Wave 0 contracts concurrently.
   Layer ordering (bronze → silver → gold) is always preserved.


---
## 5. Backfill & Reprocessing — Targeted Late-Arriving Data

**The Problem:** A partner sent corrected data for last Tuesday. You need to reload just those records without blowing away the rest of the week.

**The Solution:** `run_source(reprocess_from=..., reprocess_to=...)` lets you surgically reload a date range or specific IDs — the incremental watermark is bypassed for that run only.

In [24]:
import os
import shutil
from datetime import date, timedelta

# ── Clean slate ─────────────────────────────────────────────────────
BF_DIR = "./backfill_demo"
BF_LANDING = f"{BF_DIR}/landing"
if os.path.exists(BF_DIR):
    shutil.rmtree(BF_DIR)
os.makedirs(BF_LANDING, exist_ok=True)

# ── Contract with source.type = landing + reprocess column ─────────
backfill_contract = s.write_contract(
    """
version: 1.0.0
dataset: daily_events
info:
  title: bronze_daily_events
  target_layer: bronze

source:
  type: landing
  path: "./backfill_demo/landing/*.ndjson"
  format: ndjson

model:
  fields:
    - name: event_id
      type: integer
      required: true
    - name: event_date
      type: string
      required: true
    - name: payload
      type: string

materialization:
  reprocess_date_column: event_date

quality:
  row_rules:
    - name: has_payload
      sql: "payload IS NOT NULL"
""",
    "03_engine_scale_demo/daily_events.yaml",
)

# ── Generate a week of events and write to landing ────────────────
today = date.today()
rows = []
for i in range(7):
    day = (today - timedelta(days=6 - i)).isoformat()
    for j in range(50):
        rows.append({"event_id": i * 50 + j, "event_date": day, "payload": f"data_{i}_{j}"})

full_week = pl.DataFrame(rows)
full_week.write_ndjson(f"{BF_LANDING}/events_full_week.ndjson")
print(f"Full dataset: {len(full_week)} rows across 7 days")
print(full_week.group_by("event_date").len().sort("event_date"))

# ── Full load first ──────────────────────────────────────────────────
bf_proc = ll.DataProcessor(backfill_contract, engine=ENGINE)
g_full, b_full = bf_proc.run_source()
g_full, b_full = s.to_polars(g_full), s.to_polars(b_full)
r_full = bf_proc.last_report
print(f"\nFull load: {r_full.get('counts', {}).get('source', '?')} rows")

# ── Targeted backfill: reload just 2 days ──────────────────────────
target_start = (today - timedelta(days=3)).isoformat()
target_end = (today - timedelta(days=2)).isoformat()

g_bp, b_bp = bf_proc.run_source(
    reprocess_from=target_start,
    reprocess_to=target_end,
)
g_bp, b_bp = s.to_polars(g_bp), s.to_polars(b_bp)
r_bp = bf_proc.last_report

print(f"\n{'=' * 50}")
print(f"BACKFILL: {target_start} to {target_end}")
print(f"{'=' * 50}")
print(f"  Rows reprocessed : {r_bp.get('counts', {}).get('source', '?')}")
print(f"  Good / Bad       : {r_bp.get('counts', {}).get('good', '?')} / {r_bp.get('counts', {}).get('bad', '?')}")
print("\n\u2705 Only the targeted date range was reprocessed \u2014 rest of the week untouched.")

2026-04-28 06:49:09.210 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: backfill_demo\landing\*.ndjson via polars
2026-04-28 06:49:09.219 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 350 | Total: 350 | Good: 350 | Quarantine: 0 | Ratio: 0.00%
2026-04-28 06:49:09.221 | INFO     | lakelogic.core.processor:run_source:1374 - Loading source: backfill_demo\landing\*.ndjson via polars
2026-04-28 06:49:09.223 | INFO     | lakelogic.core.processor:run_source:1414 - Reprocessing mode: date range [2026-04-25 .. 2026-04-26] — incremental watermark bypassed
2026-04-28 06:49:09.226 | INFO     | lakelogic.core.processor:_apply_reprocess_date_filter:2596 - Reprocess filter (event_date): 350 → 100 rows [2026-04-25 .. 2026-04-26]
2026-04-28 06:49:09.231 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=bronze] | Source: 100 | Total: 100 | Good: 100 | Quarantine: 0 | Ratio: 0.00%


Full dataset: 350 rows across 7 days
shape: (7, 2)
┌────────────┬─────┐
│ event_date ┆ len │
│ ---        ┆ --- │
│ str        ┆ u32 │
╞════════════╪═════╡
│ 2026-04-22 ┆ 50  │
│ 2026-04-23 ┆ 50  │
│ 2026-04-24 ┆ 50  │
│ 2026-04-25 ┆ 50  │
│ 2026-04-26 ┆ 50  │
│ 2026-04-27 ┆ 50  │
│ 2026-04-28 ┆ 50  │
└────────────┴─────┘

Full load: 350 rows

BACKFILL: 2026-04-25 to 2026-04-26
  Rows reprocessed : 100
  Good / Bad       : 100 / ?

✅ Only the targeted date range was reprocessed — rest of the week untouched.


---
## 6. External Logic — Custom Python Hooks

**The Problem:** Your Gold-layer transformation requires 200 lines of business logic — joins, pivots, ML scoring — that won't fit in a SQL rule.

**The Solution:** Declare `external_logic` in the contract. LakeLogic calls your custom Python function, feeds it the validated DataFrame, and then applies its own quality rules and lineage to the result.

In [25]:
import os
import shutil
import yaml
import polars as pl

# ── Clean slate ─────────────────────────────────────────────────────
EXT_DIR = "03_engine_scale_demo/external_logic_demo"
if os.path.exists(EXT_DIR):
    shutil.rmtree(EXT_DIR)
os.makedirs(f"{EXT_DIR}/transforms", exist_ok=True)

# ── Step 1: Write the custom Python transform to disk ────────────
transform_code = """
import polars as pl

def run(df, *, fiscal_year=2026, include_refunds=False, **kwargs):
    \"\"\"Gold-layer aggregation - called by LakeLogic.\"\"\"\n
    # Filter out refunds if requested
    if not include_refunds:
        df = df.filter(pl.col("status") != "refunded")
    return df.group_by("region").agg(
        pl.col("amount").sum().alias("total_revenue"),
        pl.col("order_id").count().alias("order_count"),
    )
"""

transform_path = f"{EXT_DIR}/transforms/revenue_summary.py"
with open(transform_path, "w", encoding="utf-8") as f:
    f.write(transform_code.strip() + "\n")

print("transforms/revenue_summary.py")
print("─" * 60)
print(transform_code.strip())
print("─" * 60)

# ── Step 2: Create the contract referencing the script ────────────
ext_yaml = """
version: 1.0.0
dataset: gold_revenue_summary
info:
  title: gold_revenue_summary
  target_layer: gold

model:
  fields:
    - name: region
      type: string
      required: true
    - name: total_revenue
      type: float
    - name: order_count
      type: integer

external_logic:
  type: python
  path: "transforms/revenue_summary.py"
  entrypoint: run
  args:
    fiscal_year: 2026
    include_refunds: false

quality:
  row_rules:
    - name: positive_revenue
      sql: "total_revenue >= 0"
"""

ext_contract_path = s.write_contract(ext_yaml, f"{EXT_DIR}/gold_revenue_summary.yaml")

# ── Step 3: Generate realistic source data ────────────────────────
import random

random.seed(42)

regions = ["EMEA", "APAC", "Americas", "LATAM"]
statuses = ["completed", "completed", "completed", "refunded", "pending"]

source_data = pl.DataFrame(
    {
        "order_id": list(range(1, 201)),
        "region": [random.choice(regions) for _ in range(200)],
        "amount": [round(random.uniform(10, 500), 2) for _ in range(200)],
        "status": [random.choice(statuses) for _ in range(200)],
    }
)

print("\n1. SOURCE DATA (200 orders across 4 regions):\n")
display(source_data.head(5))
print(f"   ... {len(source_data)} total rows")

# ── Step 4: Run the pipeline — LakeLogic calls your script ───────
proc = ll.DataProcessor(ext_contract_path, engine=ENGINE)
result = proc.run(source_data)
result.good, result.bad = s.to_polars(result.good), s.to_polars(result.bad)

print("\n2. AFTER EXTERNAL LOGIC (your script aggregated by region):\n")
display(result.good)

print("\n" + "─" * 60)
print("What happened:")
print("  1. LakeLogic validated 200 rows against the contract schema")
print("  2. Called transforms/revenue_summary.py → run(df, fiscal_year=2026)")
print("  3. Your script filtered refunds + aggregated by region")
print("  4. LakeLogic applied quality rules (positive_revenue >= 0)")
print("  5. Lineage metadata injected automatically")
print("\n✅ Your custom logic. LakeLogic's quality rules + lineage still apply.")

transforms/revenue_summary.py
────────────────────────────────────────────────────────────
import polars as pl

def run(df, *, fiscal_year=2026, include_refunds=False, **kwargs):
    """Gold-layer aggregation - called by LakeLogic."""

    # Filter out refunds if requested
    if not include_refunds:
        df = df.filter(pl.col("status") != "refunded")
    return df.group_by("region").agg(
        pl.col("amount").sum().alias("total_revenue"),
        pl.col("order_id").count().alias("order_count"),
    )
────────────────────────────────────────────────────────────

1. SOURCE DATA (200 orders across 4 regions):



order_id,region,amount,status
i64,str,f64,str
1,"""EMEA""",34.79,"""pending"""
2,"""EMEA""",499.65,"""pending"""
3,"""Americas""",419.65,"""completed"""
4,"""APAC""",484.81,"""refunded"""
5,"""APAC""",463.92,"""refunded"""


2026-04-28 06:49:09.272 | INFO     | lakelogic.core.processor:run:818 - Run complete [layer=gold] | Source: 4 | Total: 4 | Good: 4 | Quarantine: 0 | Ratio: 0.00%


   ... 200 total rows

2. AFTER EXTERNAL LOGIC (your script aggregated by region):



region,total_revenue,order_count
str,f64,i64
"""EMEA""",10692.22,43
"""LATAM""",8706.3,34
"""Americas""",9414.46,33
"""APAC""",11213.89,39



────────────────────────────────────────────────────────────
What happened:
  1. LakeLogic validated 200 rows against the contract schema
  2. Called transforms/revenue_summary.py → run(df, fiscal_year=2026)
  3. Your script filtered refunds + aggregated by region
  4. LakeLogic applied quality rules (positive_revenue >= 0)
  5. Lineage metadata injected automatically

✅ Your custom logic. LakeLogic's quality rules + lineage still apply.


## What You Just Saw

| # | Feature | How |
|---|---------|-----|
| 1 | **Engine portability** | Same contract on Polars and DuckDB, identical results |
| 2 | **Dimensional modeling** | `strategy: scd2` — full history tracking declared in YAML |
| 3 | **Incremental processing** | `pipeline_log` watermark — only new files are loaded |
| 4 | **Parallel processing** | `pipeline.run(parallel=True)` — concurrent wave execution |
| 5 | **Backfill & reprocessing** | `reprocess_from`/`reprocess_to` — surgical date-range reload |
| 6 | **External logic** | `external_logic.type: python` — custom transforms with full lineage |

---
## Go Deeper — Explore by Capability

Each notebook below maps to a pillar of LakeLogic's [Technical Capabilities](https://lakelogic.github.io/LakeLogic/#technical-capabilities):

| # | Notebook | What You'll See |
|---|---|---|
| 🛡️ | **[Data Quality & Trust](01_data_quality_trust.ipynb)** | Reconciliation proofs, Pydantic validation, SQL-first rules, SLO monitoring |
| 📜 | **[Compliance & Governance](02_compliance_governance.ipynb)** | GDPR erasure in 2 lines, automatic lineage, cost intelligence |
| ⚡ | **[Engine & Scale](03_engine_scale.ipynb)** | Same contract on Polars & DuckDB, incremental processing, dry run |
| 🔧 | **[Developer Experience](04_developer_experience.ipynb)** | Structured diagnostics, DDL generation, surgical resets, multi-channel alerts |
| 🧬 | **[Data Generation & AI](05_data_generation_ai.ipynb)** | Synthetic data, referential integrity, edge case injection, contract inference |
| 🔌 | **[Integrations](06_integrations.ipynb)** | dbt adapter, dlt sources, contract-driven quality gates on arrival |

> **Each notebook is self-contained** — pick the capability that matters most to you and run it independently.

## 🧊 Multi-Format Materialization: Apache Iceberg

LakeLogic is table-format agnostic. While Delta Lake is the default, **Apache Iceberg** is rapidly becoming the open standard for analytical tables (especially across Snowflake, AWS, and Databricks workflows). You can natively output validated data directly into Iceberg format using either DuckDB or Spark simply by changing the `format` flag in your contract materialization block.

In [26]:
iceberg_contract_yaml = """
name: customer_iceberg_table
version: "1.0"
system: crm
schema:
  - name: user_id
    type: integer
    not_null: true
  - name: email
    type: string

materialization:
  format: iceberg
  target_path: "/tmp/lakelogic/iceberg/customers"
"""

# Write the contract to disk
with open("iceberg_contract.yaml", "w") as f:
    f.write(iceberg_contract_yaml)

print("✅ Iceberg Data Contract generated: iceberg_contract.yaml")

# Note: Running this pipeline locally with DuckDB engine will automatically
# download and install the DuckDB Iceberg extension and write the table locally.

✅ Iceberg Data Contract generated: iceberg_contract.yaml
